- **Name:** 14.1_dataframe_windfunc_ranking
- **Author:** Shamas Imran
- **Desciption:** Ranking functions using DataFrame API with window functions
- **Date:** 19-Aug-2025
<!--
REVISION HISTORY
Version          Date        Author           Desciption
01           19-Aug-2025   Shamas Imran       Implemented rank, dense_rank, row_number  
                                              Partitioned data using Window spec  
                                              Compared ranking function outputs  
-->

In [1]:
from pyspark.sql.types import *
from pyspark.sql import Row
from datetime import date
import random

# -------------------------------
# 1. Student DataFrame
# -------------------------------
student_schema = StructType([
    StructField('StudentID', IntegerType(), False),
    StructField('StudentName', StringType(), True),
    StructField('StudentAge', IntegerType(), True)
])

student_data = [
    (1, "Alice", 34), 
    (2, "Bob", 45), 
    (3, "Charlie", 29),
    (4, "Shamas", 40)
]

df_student = spark.createDataFrame(student_data, student_schema)

# -------------------------------
# 2. Course DataFrame
# -------------------------------
course_schema = StructType([
    StructField('CourseID', IntegerType(), False),
    StructField('CourseName', StringType(), True),
    StructField('CourseTitle', StringType(), True),
])

course_data = [
    (1, "Physics", "1111"), 
    (2, "Chemistry", "2222"), 
    (3, "English", "3333"),
    (4, "Computer Science", "4444")
]

df_course = spark.createDataFrame(course_data, course_schema)

# -------------------------------
# 3. Enrollment DataFrame
# -------------------------------
enrollment_schema = StructType([
    StructField("EnrollmentID", IntegerType(), False),
    StructField("StudentID_FK", IntegerType(), False),
    StructField("CourseID_FK", IntegerType(), False),
    StructField("EnrollmentDate", DateType(), True)
])

enrollment_data = [
    (1, 1, 1, date(2023, 9, 1)),   # Alice -> Physics
    (2, 2, 2, date(2023, 9, 2)),   # Bob -> Chemistry
    (3, 4, 4, date(2023, 9, 4)),   # Shamas -> Computer Science
    (4, 1, 2, date(2023, 9, 5)),   # Alice -> Chemistry
]

df_enrollment = spark.createDataFrame(enrollment_data, enrollment_schema)

# -------------------------------
# 4. Score DataFrame
# -------------------------------
semesters = ["2023-Spring", "2023-Fall", "2024-Spring", "2024-Fall", "2025-Spring"]
score_data = []
score_id = 1
enrollment_ids = [row.EnrollmentID for row in df_enrollment.collect()]

for enrollment_id in enrollment_ids:
    selected_semesters = random.sample(semesters, k=random.randint(2, 4))
    for sem in selected_semesters:
        score_data.append(Row(
            ScoreID=score_id,
            EnrollmentID_FK=enrollment_id,
            Semester=sem,
            Score=random.randint(60, 100)
        ))
        score_id += 1

score_schema = StructType([
    StructField("ScoreID", IntegerType(), False),
    StructField("EnrollmentID_FK", IntegerType(), False),
    StructField("Semester", StringType(), True),
    StructField("Score", IntegerType(), True)
])

df_score = spark.createDataFrame(score_data, schema=score_schema)

StatementMeta(, 14d44f66-ff51-42a5-bae5-9a9f470e8f35, 3, Finished, Available, Finished)

In [2]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

# Define window: per course, order by score descending
course_window = Window.partitionBy("e.CourseID_FK").orderBy(F.col("s.Score").desc())

StatementMeta(, 14d44f66-ff51-42a5-bae5-9a9f470e8f35, 4, Finished, Available, Finished)

In [3]:
# 1. Rank students within each course by score
df_score.alias("s") \
    .join(df_enrollment.alias("e"), F.col("s.EnrollmentID_FK") == F.col("e.EnrollmentID")) \
    .join(df_student.alias("st"), F.col("e.StudentID_FK") == F.col("st.StudentID")) \
    .withColumn("Rank", F.rank().over(course_window)) \
    .select("st.StudentName", "e.CourseID_FK", "s.Score", "Rank") \
    .show()

StatementMeta(, 14d44f66-ff51-42a5-bae5-9a9f470e8f35, 5, Finished, Available, Finished)

+-----------+-----------+-----+----+
|StudentName|CourseID_FK|Score|Rank|
+-----------+-----------+-----+----+
|      Alice|          1|   98|   1|
|      Alice|          1|   91|   2|
|      Alice|          1|   75|   3|
|      Alice|          1|   71|   4|
|      Alice|          2|   96|   1|
|      Alice|          2|   74|   2|
|        Bob|          2|   68|   3|
|        Bob|          2|   64|   4|
|     Shamas|          4|   83|   1|
|     Shamas|          4|   82|   2|
|     Shamas|          4|   66|   3|
|     Shamas|          4|   61|   4|
+-----------+-----------+-----+----+



In [0]:
# 2. Dense rank (no gaps in ranking)
print("Dense Rank of Students by Score within each Course")
df_score.alias("s") \
    .join(df_enrollment.alias("e"), F.col("s.EnrollmentID_FK") == F.col("e.EnrollmentID")) \
    .join(df_student.alias("st"), F.col("e.StudentID_FK") == F.col("st.StudentID")) \
    .withColumn("DenseRank", F.dense_rank().over(course_window)) \
    .select("st.StudentName", "e.CourseID_FK", "s.Score", "DenseRank") \
    .show()

In [0]:
# 3. Row number (unique sequence within partition)
df_score.alias("s") \
    .join(df_enrollment.alias("e"), F.col("s.EnrollmentID_FK") == F.col("e.EnrollmentID")) \
    .join(df_student.alias("st"), F.col("e.StudentID_FK") == F.col("st.StudentID")) \
    .withColumn("RowNum", F.row_number().over(course_window)) \
    .select("st.StudentName", "e.CourseID_FK", "s.Score", "RowNum") \
    .show()

In [0]:

# 4. Top scorer per course (filtering rank = 1)
df_score.alias("s") \
    .join(df_enrollment.alias("e"), F.col("s.EnrollmentID_FK") == F.col("e.EnrollmentID")) \
    .join(df_student.alias("st"), F.col("e.StudentID_FK") == F.col("st.StudentID")) \
    .withColumn("Rank", F.rank().over(course_window)) \
    .filter(F.col("Rank") == 1) \
    .select("st.StudentName", "e.CourseID_FK", "s.Score") \
    .show()